In [1]:
pip install pyspark==3.5.1


  Using cached pyspark-3.5.1-py2.py3-none-any.whl
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.1.1
    Can't uninstall 'pyspark'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyspark-connect 4.1.1 requires pyspark==4.1.1, but you have pyspark 3.5.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys

print("🔥 1. CƯỠNG CHẾ XÓA PYSPARK 4.1.1 KHỎI BỘ NHỚ RAM...")
# Quét toàn bộ các module pyspark/py4j đang chạy ngầm và xóa sổ chúng
modules_to_remove = [mod for mod in sys.modules if mod.startswith('pyspark') or mod.startswith('py4j')]
for mod in modules_to_remove:
    del sys.modules[mod]

print("🧹 2. DỌN SẠCH ĐƯỜNG DẪN CŨ...")
sys.path = [p for p in sys.path if "/usr/local/spark" not in p]
if "PYTHONPATH" in os.environ:
    del os.environ["PYTHONPATH"]

print("🎯 3. ÉP ƯU TIÊN SỐ 1 CHO PYSPARK 3.5.1...")
conda_site_packages = "/opt/conda/lib/python3.13/site-packages"
# Dùng insert(0, ...) để chèn lên đầu danh sách, bắt Python phải đọc thư mục này đầu tiên
if conda_site_packages not in sys.path:
    sys.path.insert(0, conda_site_packages)

os.environ["SPARK_HOME"] = os.path.join(conda_site_packages, "pyspark")
os.environ["PYSPARK_PYTHON"] = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"

# Bây giờ import lại từ đầu
import pyspark

print("\n================ KẾT QUẢ ================")
print(f"✅ PySpark version đang chạy: {pyspark.__version__}")
print(f"✅ Đường dẫn thực tế: {pyspark.__file__}")

if pyspark.__version__.startswith("3.5"):
    print("🎉 THÀNH CÔNG! BẠN ĐÃ THOÁT KHỎI 4.1.1!")
else:
    print("💀 Kernel này được hardcode quá sâu ở cấp độ C/Java.")

🔥 1. CƯỠNG CHẾ XÓA PYSPARK 4.1.1 KHỎI BỘ NHỚ RAM...
🧹 2. DỌN SẠCH ĐƯỜNG DẪN CŨ...
🎯 3. ÉP ƯU TIÊN SỐ 1 CHO PYSPARK 3.5.1...

================ KẾT QUẢ ================
✅ PySpark version đang chạy: 3.5.1
✅ Đường dẫn thực tế: /opt/conda/lib/python3.13/site-packages/pyspark/__init__.py
🎉 THÀNH CÔNG! BẠN ĐÃ THOÁT KHỎI 4.1.1!


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, split 
import pandas as pd
import matplotlib.pyplot as plt
import re
# --- CẤU HÌNH SPARK ĐỂ ĐỌC MINIO + ICEBERG ---
print("Đang khởi tạo SparkSession...")
spark = SparkSession.builder \
    .appName("MinIOLogAnalysis_Iceberg") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.secret.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "hadoop") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://raw-financial-data/iceberg_warehouse") \
    .getOrCreate()

print("✅ Khởi tạo Spark + Iceberg thành công!")

Đang khởi tạo SparkSession...
✅ Khởi tạo Spark + Iceberg thành công!


In [4]:
print("\n🛠️ Đang rà soát và vá lỗi cấu hình thời gian của Hadoop...")
hadoop_conf = spark._jsc.hadoopConfiguration()
iterator = hadoop_conf.iterator()
keys_to_fix = {}

while iterator.hasNext():
    entry = iterator.next()
    val = str(entry.getValue()).strip().lower()
    
    # Tìm dạng số đi kèm s, m, h, d
    match = re.fullmatch(r"(\d+)([smhd])", val)
    if match:
        num = int(match.group(1))
        unit = match.group(2)
        
        # Quy đổi về mili-giây
        if unit == 's': ms_val = num * 1000
        elif unit == 'm': ms_val = num * 60 * 1000
        elif unit == 'h': ms_val = num * 3600 * 1000
        elif unit == 'd': ms_val = num * 86400 * 1000
        
        keys_to_fix[entry.getKey()] = str(ms_val)

for key, new_val in keys_to_fix.items():
    print(f"   🔧 Vá lỗi: {key} (Đổi '{hadoop_conf.get(key)}' thành '{new_val}')")
    hadoop_conf.set(key, new_val)

print("✅ Đã dọn sạch mọi mầm mống lỗi thời gian (s, m, h, d)!\n")


🛠️ Đang rà soát và vá lỗi cấu hình thời gian của Hadoop...
   🔧 Vá lỗi: fs.s3a.s3guard.consistency.retry.interval (Đổi '2s' thành '2000')
   🔧 Vá lỗi: fs.s3a.metadatastore.metadata.ttl (Đổi '15m' thành '900000')
   🔧 Vá lỗi: hadoop.security.groups.shell.command.timeout (Đổi '0s' thành '0')
   🔧 Vá lỗi: hadoop.service.shutdown.timeout (Đổi '30s' thành '30000')
   🔧 Vá lỗi: fs.s3a.multipart.threshold (Đổi '128M' thành '7680000')
   🔧 Vá lỗi: yarn.resourcemanager.delegation-token-renewer.thread-retry-interval (Đổi '60s' thành '60000')
   🔧 Vá lỗi: fs.s3a.block.size (Đổi '32M' thành '1920000')
   🔧 Vá lỗi: fs.s3a.multipart.size (Đổi '64M' thành '3840000')
   🔧 Vá lỗi: fs.azure.sas.expiry.period (Đổi '90d' thành '7776000000')
   🔧 Vá lỗi: yarn.resourcemanager.delegation-token-renewer.thread-timeout (Đổi '60s' thành '60000')
   🔧 Vá lỗi: fs.s3a.assumed.role.session.duration (Đổi '30m' thành '1800000')
✅ Đã dọn sạch mọi mầm mống lỗi thời gian (s, m, h, d)!



In [5]:
from pyspark.sql.functions import col, to_date

# --- ĐẢM BẢO NAMESPACE TỒN TẠI TRƯỚC KHI GHI ---
spark.sql("CREATE NAMESPACE IF NOT EXISTS my_catalog.raw_zone")

# ====================================================================
# 📰 PIPELINE 1: XỬ LÝ DỮ LIỆU TIN TỨC (NEWS)
# ====================================================================
print("🚀 ĐANG XỬ LÝ THƯ MỤC TIN TỨC (news_data)...")
path_news = "s3a://raw-financial-data/raw_zone/news_data/"

# 1. Đọc dữ liệu
df_news_raw = spark.read \
    .format("json") \
    .option("multiLine", "true") \
    .option("recursiveFileLookup", "true") \
    .option("mode", "DROPMALFORMED") \
    .option("inferSchema", "true") \
    .load(path_news)

# 2. Bóc tách
df_news_clean = df_news_raw.select(
    col("id"),
    col("content.title").alias("title"),
    col("content.summary").alias("summary"),
    col("content.pubDate").alias("published_at")
)

# 3. Ghi vào Iceberg
df_news_clean.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable("my_catalog.raw_zone.news_data_clean")

print(f"   ✅ Đã lưu xong bảng News! (Tổng: {df_news_clean.count()} bài báo)")


# ====================================================================
# 📈 PIPELINE 2: XỬ LÝ DỮ LIỆU CHỨNG KHOÁN (MARKET)
# ====================================================================
print("\n🚀 ĐANG XỬ LÝ THƯ MỤC CHỨNG KHOÁN (market_data)...")
path_market = "s3a://raw-financial-data/raw_zone/market_data/"

# 1. Đọc dữ liệu
df_market_raw = spark.read \
    .format("json") \
    .option("multiLine", "true") \
    .option("recursiveFileLookup", "true") \
    .option("mode", "DROPMALFORMED") \
    .option("inferSchema", "true") \
    .load(path_market)

# 2. Chuẩn hóa & Đổi tên cột
df_market_clean = df_market_raw.select(
    to_date(col("Date")).alias("trade_date"),
    col("Open").alias("open_price"),
    col("High").alias("high_price"),
    col("Low").alias("low_price"),
    col("Close").alias("close_price"),
    col("Volume").alias("volume"),
    col("Dividends").alias("dividends"),
    col("`Stock Splits`").alias("stock_splits"),
    col("`Capital Gains`").alias("capital_gains")
)

# 3. Ghi vào Iceberg
df_market_clean.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable("my_catalog.raw_zone.market_data_clean")

print(f"   ✅ Đã lưu xong bảng Market! (Tổng: {df_market_clean.count()} phiên giao dịch)")

print("\n🎉 HOÀN TẤT XUẤT SẮC! DATA LAKE CỦA BẠN ĐÃ SẴN SÀNG ĐỂ TRUY VẤN!")

🚀 ĐANG XỬ LÝ THƯ MỤC TIN TỨC (news_data)...
   ✅ Đã lưu xong bảng News! (Tổng: 100 bài báo)

🚀 ĐANG XỬ LÝ THƯ MỤC CHỨNG KHOÁN (market_data)...
   ✅ Đã lưu xong bảng Market! (Tổng: 2493 phiên giao dịch)

🎉 HOÀN TẤT XUẤT SẮC! DATA LAKE CỦA BẠN ĐÃ SẴN SÀNG ĐỂ TRUY VẤN!


In [7]:
print("🗂️ 1. DANH SÁCH CÁC NAMESPACE (DATABASE) TRONG CATALOG:")
spark.sql("SHOW DATABASES IN my_catalog").show(truncate=False)

print("\n📊 2. DANH SÁCH CÁC BẢNG TRONG RAW_ZONE:")
spark.sql("SHOW TABLES IN my_catalog.raw_zone").show(truncate=False)


🗂️ 1. DANH SÁCH CÁC NAMESPACE (DATABASE) TRONG CATALOG:
+--------------+
|namespace     |
+--------------+
|processed_zone|
|raw_zone      |
+--------------+


📊 2. DANH SÁCH CÁC BẢNG TRONG RAW_ZONE:
+---------+-----------------+-----------+
|namespace|tableName        |isTemporary|
+---------+-----------------+-----------+
|raw_zone |news_data_clean  |false      |
|raw_zone |market_data_clean|false      |
+---------+-----------------+-----------+



In [8]:
print("\n🔍 3. XEM THỬ 5 DÒNG DỮ LIỆU ĐẦU TIÊN TRONG BẢNG:")

# Cách 1: Dùng SQL thuần
spark.sql("""
    SELECT *
    FROM my_catalog.raw_zone.news_data_clean
    LIMIT 5
""").show(truncate=False)


print("\n🔍 3. XEM THỬ 5 DÒNG DỮ LIỆU ĐẦU TIÊN TRONG BẢNG:")

# Cách 1: Dùng SQL thuần
spark.sql("""
    SELECT *
    FROM my_catalog.raw_zone.market_data_clean
    LIMIT 5
""").show(truncate=False)




🔍 3. XEM THỬ 5 DÒNG DỮ LIỆU ĐẦU TIÊN TRONG BẢNG:
+------------------------------------+--------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+
|id                                  |title                                                                           |summary                                                                                                                                                                                         

In [12]:
pip install spacy


Note: you may need to restart the kernel to use updated packages.


In [14]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 13.7 MB/s  0:00:01 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [15]:
import re
import spacy
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StructType, StructField, ArrayType, StringType

# Khởi tạo spaCy
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

# Định nghĩa cấu trúc trả về: Một bộ chứa 2 mảng riêng biệt
nlp_schema = StructType([
    StructField("tokens", ArrayType(StringType()), False),
    StructField("lemmas", ArrayType(StringType()), False)
])

def extract_tokens_and_lemmas(text):
    """
    Chạy spaCy 1 lần, bóc tách song song cả Token gốc và Lemma.
    Trả về dạng Dictionary để Spark tự động map vào StructType.
    """
    if not text:
        return {"tokens": [], "lemmas": []}
        
    # Làm sạch cơ bản
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    if not text:
        return {"tokens": [], "lemmas": []}
        
    doc = nlp(text)
    
    tokens_list = []
    lemmas_list = []
    
    for token in doc:
        # Lọc stopwords và từ quá ngắn
        if not token.is_stop and len(token.text.strip()) > 1:
            tokens_list.append(token.text)     # Lấy từ gốc
            lemmas_list.append(token.lemma_)   # Lấy từ nguyên thể
            
    return {"tokens": tokens_list, "lemmas": lemmas_list}

# Đăng ký UDF với schema kép
dual_nlp_udf = udf(extract_tokens_and_lemmas, nlp_schema)

In [16]:
print("Đang xử lý NLP song song cho cả Title và Summary...")

# 1. Gọi UDF cho cả 2 cột
df_processed = df_news_clean \
    .withColumn("title_nlp", dual_nlp_udf(col("title"))) \
    .withColumn("summary_nlp", dual_nlp_udf(col("summary")))

# 2. Bóc tách toàn bộ ra thành các cột phẳng (Flat columns)
df_final = df_processed.select(
    col("id"),
    col("published_at"), # Giữ lại ngày đăng để mốt filter nếu cần
    
    # Bóc tách Title
    col("title_nlp.tokens").alias("title_tokens"),
    col("title_nlp.lemmas").alias("title_lemmas"),
    
    # Bóc tách Summary
    col("summary_nlp.tokens").alias("summary_tokens"),
    col("summary_nlp.lemmas").alias("summary_lemmas")
)

# Xem thử kết quả
df_final.select("id", "title_lemmas", "summary_lemmas").show(5, truncate=50)
# Xem thử kết quả
df_final.select("id", "title_tokens", "summary_tokens").show(5, truncate=50)

Đang xử lý NLP song song cho cả Title và Summary...
+------------------------------------+--------------------------------------------------+--------------------------------------------------+
|                                  id|                                      title_lemmas|                                    summary_lemmas|
+------------------------------------+--------------------------------------------------+--------------------------------------------------+
|47d6de09-8456-4330-9121-a92c931adb53|[nvidia, partner, microsoft, new, rtx, spark, l...|[nvidia, ceo, jensen, huang, introduce, new, rt...|
|48e2edc1-8e96-434b-8bdf-fb0f28451d4c|       [value, nvidia, new, chip, add, window, pc]|[nvidia, stock, nvda, cruise, high, start, mond...|
|da795e64-6fad-4c7a-9bec-9732d692f2f4|[meta, alphabet, amazon, microsoft, get, hook, ...|             [debt, big, tool, big, tech, arsenal]|
|7b520322-5eda-3b78-b880-13b922d0c113|[microsoft, nvidia, rtx, spark, put, ai, pc, in...|[nvidia, micr

In [ ]:
# # Bảng 1: Dành cho team cần Token gốc
# df_final.select("id", "published_at", "title_tokens", "summary_tokens") \
#     .write.format("iceberg").mode("overwrite") \
#     .saveAsTable("my_catalog.processed_zone.news_tokens")

# # Bảng 2: Dành cho team cần Lemma
# df_final.select("id", "published_at", "title_lemmas", "summary_lemmas") \
#     .write.format("iceberg").mode("overwrite") \
#     .saveAsTable("my_catalog.processed_zone.news_lemmas")